# Notebook 1 — Adversarial Attack Generation

**Pipeline position:** this is stage 1 of 5. It produces the `{clean, adv, noisy, label}`
tensor triplets that every downstream notebook (Mahalanobis/LID extraction, layer
detectors, the ENAD ensemble + GA) consumes. Nothing here fits a detector — it only
*generates the data* the detectors are trained and evaluated on.

For each batch it keeps only samples that are **(a) correctly classified when clean,
(b) still correct under Gaussian noise, and (c) misclassified after the attack** — i.e.
successful attacks on originally-correct, noise-robust inputs. This filtering (`selected`)
is what makes the detection task well-posed.

---
### Attribution 
This notebook builds on prior work and is **not** all original:

- **ResNet-34 architecture + dataset loaders + pretrained weights** — Lee et al.,
  *A Simple Unified Framework for Detecting Out-of-Distribution Samples and Adversarial
  Attacks* (NeurIPS 2018), repo `pokaxpoka/deep_Mahalanobis_detector`. Imported directly so
  the architecture exactly matches the released weights.
- **DeepFool / CW-L2 attack implementations** — adapted (with only deprecation fixes) from
  the same lineage.

The contribution of *this project* lives in later notebooks (The design of supervised detectors and GA-optimized ensemble implementation) — **not** in attack generation itself.

---
### Requirements to run on Kaggle
- **Accelerator:** GPU (Settings → Accelerator → GPU).
- **Internet:** ON (needed for the `git clone` in the setup cell).
- **Pretrained weights** attached as a dataset at `/kaggle/input/datasets/resnet-pth/` containing
  `resnet_cifar10.pth`, `resnet_svhn.pth`, `resnet_cifar100.pth` (adjust `weights_dir` if
  your path differs).

### Output 
Writes to `/kaggle/working/{DS_UPPER}/{ADV}/` files named
`clean_data_{net}_{ds}_{adv}.pth`, `adv_data_...`, `noisy_data_...`, `label_...`.
After running, **Save Version → publish the output as a Kaggle Dataset** named e.g.
`attacked-pth-files`, then *Add data* it into Notebook 2.

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

REPO = "deep_Mahalanobis_detector"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "--quiet",
         "https://github.com/pokaxpoka/deep_Mahalanobis_detector.git"],
        check=True,
    )
sys.path.append("./" + REPO)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T

from deep_Mahalanobis_detector import models, data_loader   \

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

Device: cuda | torch 2.10.0+cu128


In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    ds_name: str  = "cifar10"          # 'cifar10' | 'cifar100' | 'svhn'
    net_type: str = "resnet"           
    adv_type: str = "FGSM"             # 'FGSM' | 'BIM' | 'DeepFool' | 'CWL2'
    batch_size: int = 200
    data_root: str  = "/kaggle/working/data"
    weights_dir: str = "/kaggle/input/datasets/sealeopard/resnet-pth"  
    out_root: str    = "/kaggle/working"        

MIN_PIXEL, MAX_PIXEL = -2.42906570435, 2.75373125076
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

def num_classes_of(ds_name):
    return 100 if ds_name == "cifar100" else 10

def attack_params(ds_name, adv_type):
    """Per-attack step size (adv_noise) and Gaussian noise scale, mirroring the
    original ResNet configuration. adv_noise is unused by CW-L2 (it has its own lr)."""
    adv_noise = {
        "FGSM": 0.05,
        "BIM":  0.01,
        "DeepFool": {"cifar10": 0.18, "cifar100": 0.03, "svhn": 0.10}.get(ds_name, 0.10),
        "CWL2": 0.0,
    }[adv_type]
    rns = {
        "FGSM":     {"cifar10": 0.25/4, "cifar100": 0.25/8, "svhn": 0.25/4},
        "BIM":      {"cifar10": 0.13/2, "cifar100": 0.13/4, "svhn": 0.13/2},
        "DeepFool": {"cifar10": 0.25/4, "cifar100": 0.13/4, "svhn": 0.126},
        "CWL2":     {"cifar10": 0.05/2, "cifar100": 0.05/2, "svhn": 0.05/1},
    }[adv_type].get(ds_name, 0.05)
    return adv_noise, rns

cfg = Config()
print(cfg)

Config(ds_name='cifar10', net_type='resnet', adv_type='FGSM', batch_size=200, data_root='/kaggle/working/data', weights_dir='/kaggle/input/datasets/sealeopard/resnet-pth', out_root='/kaggle/working')


In [ ]:
def load_model_and_transform(cfg):
    nc = num_classes_of(cfg.ds_name)
    model = models.ResNet34(num_c=nc)
    ckpt = os.path.join(cfg.weights_dir, f"{cfg.net_type}_{cfg.ds_name}.pth")
    state = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    return model, tf, nc

def get_test_loader(cfg, tf):
    _, test_loader = data_loader.getTargetDataSet(
        cfg.ds_name, cfg.batch_size, tf, cfg.data_root
    )
    return test_loader

In [ ]:
_STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)

def _signed_scaled_grad(grad):
    sign = (grad >= 0).float() * 2 - 1        
    return sign / _STD_T

def fgsm(model, x, y, eps):
    x = x.clone().detach().to(DEVICE).requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    grad = torch.autograd.grad(loss, x)[0]
    x_adv = x.detach() + eps * _signed_scaled_grad(grad)
    return torch.clamp(x_adv, MIN_PIXEL, MAX_PIXEL).detach()

def bim(model, x, y, eps, steps=5):
    x_adv = x.clone().detach().to(DEVICE)
    for _ in range(steps):
        x_adv.requires_grad_(True)
        loss = F.cross_entropy(model(x_adv), y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = (x_adv.detach() + eps * _signed_scaled_grad(grad))
        x_adv = torch.clamp(x_adv, MIN_PIXEL, MAX_PIXEL).detach()
    return x_adv

In [ ]:
@torch.no_grad()
def _predict(model, x):
    return model(x).argmax(1)

def _deepfool_single(model, img, true_label, n_classes, max_iter=50, step_size=0.1):
    x_adv = img.clone().detach().to(DEVICE)          
    true_label = int(true_label)
    pred = true_label
    for _ in range(max_iter):
        x_in = x_adv.unsqueeze(0).detach()           
        with torch.no_grad():
            output = model(x_in).clone()             
        x_batch = x_in.repeat(n_classes, 1, 1, 1).clone().detach().requires_grad_(True)
        out_batch = model(x_batch)                   
        idx = torch.arange(n_classes, device=DEVICE)
        loss = F.nll_loss(out_batch, idx)
        grad = torch.autograd.grad(loss, x_batch)[0] 
        grad_input = -grad
        f = (output - output[0, true_label]).squeeze(0)             
        w = grad_input - grad_input[true_label].unsqueeze(0)       
        w_norm = w.view(n_classes, -1).norm(2, dim=1)               
        ratio = f.abs() / (w_norm + 1e-12)
        ratio[true_label] = float("inf")
        min_ratio, min_idx = ratio.min(0)
        ri = (min_ratio / (w_norm[min_idx] + 1e-12)) * step_size * w[min_idx]
        x_adv = (x_adv + ri).detach()
        pred = int(_predict(model, x_adv.unsqueeze(0)).item())
        if pred != true_label:
            break
    return x_adv

def deepfool(model, x, y, n_classes, step_size=0.1, max_iter=50):
    outs = []
    for i in range(x.size(0)):
        outs.append(_deepfool_single(model, x[i], int(y[i]), n_classes,
                                     max_iter=max_iter, step_size=step_size))
    return torch.stack(outs, 0)

def cw_l2(model, x, y_true, c=1.0, kappa=0.0, max_iter=100, lr=0.01):
    x = x.clone().detach().to(DEVICE)
    y_true = y_true.to(DEVICE)
    w = x.clone().detach().requires_grad_(True)
    best_w = x.clone().detach()
    best_loss = float("inf")
    opt = torch.optim.Adam([w], lr=lr)

    with torch.no_grad():
        top2 = F.softmax(model(x), dim=1).topk(2, dim=1).indices   
    argmax = top2[:, 0].clone()
    same = argmax.eq(y_true)
    argmax[same] = top2[same, 1]                                  

    for _ in range(max_iter):
        opt.zero_grad()
        recon = ((w - x) ** 2).sum()
        out = model(w)
       
        margin = out.gather(1, y_true.view(-1, 1)).squeeze(1) \
                 - out.gather(1, argmax.view(-1, 1)).squeeze(1)
        adv = c * torch.clamp(margin + kappa, min=0).sum()
        loss = recon + adv
        loss.backward()
        opt.step()
        with torch.no_grad():
            total = loss.item()
            if total < best_loss:
                best_loss = total
                best_w = w.detach().clone()
    return best_w


In [ ]:
def generate(cfg, verbose=True):
    model, tf, nc = load_model_and_transform(cfg)
    test_loader = get_test_loader(cfg, tf)
    adv_noise, rns = attack_params(cfg.ds_name, cfg.adv_type)

    clean_all, adv_all, noisy_all, label_all = [], [], [], []
    n_ok = n_adv_ok = n_noise_ok = total = 0
    gen_noise = 0.0
    selected, sel_idx = [], 0

    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)

        with torch.no_grad():
            eq = _predict(model, data).eq(target).cpu()
        n_ok += int(eq.sum())

        noisy = torch.clamp(data + rns * torch.randn_like(data), MIN_PIXEL, MAX_PIXEL)

        if cfg.adv_type == "FGSM":
            adv = fgsm(model, data, target, adv_noise)
        elif cfg.adv_type == "BIM":
            adv = bim(model, data, target, adv_noise, steps=5)
        elif cfg.adv_type == "DeepFool":
            adv = deepfool(model, data, target.cpu(), nc, step_size=adv_noise)
        elif cfg.adv_type == "CWL2":
            adv = cw_l2(model, data, target)
        else:
            raise ValueError(cfg.adv_type)
        adv = torch.clamp(adv.to(DEVICE), MIN_PIXEL, MAX_PIXEL)

        gen_noise += (data - adv).abs().flatten(1).max(1).values.sum().item()

        with torch.no_grad():
            eq_adv   = _predict(model, adv).eq(target).cpu()
            eq_noise = _predict(model, noisy).eq(target).cpu()
        n_adv_ok   += int(eq_adv.sum())
        n_noise_ok += int(eq_noise.sum())

        clean_all.append(data.cpu()); adv_all.append(adv.cpu())
        noisy_all.append(noisy.cpu()); label_all.append(target.cpu())

        for i in range(data.size(0)):
            if bool(eq[i]) and bool(eq_noise[i]) and not bool(eq_adv[i]):
                selected.append(sel_idx)
            sel_idx += 1
        total += data.size(0)

    clean = torch.cat(clean_all); adv = torch.cat(adv_all)
    noisy = torch.cat(noisy_all); label = torch.cat(label_all)
    sel = torch.as_tensor(selected, dtype=torch.long)
    clean, adv, noisy, label = clean[sel], adv[sel], noisy[sel], label[sel]

    if verbose:
        print(f"[{cfg.ds_name} / {cfg.adv_type}]")
        print(f"  clean acc  : {100.*n_ok/total:.4f}%")
        print(f"  noisy acc  : {100.*n_noise_ok/total:.4f}%")
        print(f"  adv   acc  : {100.*n_adv_ok/total:.4f}%  (lower = stronger attack)")
        print(f"  mean Linf perturbation: {gen_noise/total:.4f}")
        print(f"  kept (clean-ok & noise-ok & adv-fooled): {len(sel)} / {total}")
    return clean, adv, noisy, label

In [7]:
def save_artifacts(cfg, clean, adv, noisy, label):
    out_dir = os.path.join(cfg.out_root, cfg.ds_name.upper(), cfg.adv_type)
    os.makedirs(out_dir, exist_ok=True)
    tag = f"{cfg.net_type}_{cfg.ds_name}_{cfg.adv_type}"
    torch.save(clean, os.path.join(out_dir, f"clean_data_{tag}.pth"))
    torch.save(adv,   os.path.join(out_dir, f"adv_data_{tag}.pth"))
    torch.save(noisy, os.path.join(out_dir, f"noisy_data_{tag}.pth"))
    torch.save(label, os.path.join(out_dir, f"label_{tag}.pth"))
    print("saved ->", out_dir)
    return out_dir

In [8]:
#clean, adv, noisy, label = generate(cfg)
#save_artifacts(cfg, clean, adv, noisy, label)

In [ ]:
for adv_type in ["FGSM", "BIM", "DeepFool", "CWL2"]:
     cfg.adv_type = adv_type
     c, a, n, l = generate(cfg)
     save_artifacts(cfg, c, a, n, l)

100%|██████████| 170M/170M [39:10<00:00, 72.5kB/s]


[cifar10 / FGSM]
  clean acc  : 93.67%
  noisy acc  : 92.56%
  adv   acc  : 23.98%  (lower = stronger attack)
  mean Linf perturbation: 0.2508
  kept (clean-ok & noise-ok & adv-fooled): 6770 / 10000
saved -> /kaggle/working/CIFAR10/FGSM
[cifar10 / BIM]
  clean acc  : 93.67%
  noisy acc  : 92.26%
  adv   acc  : 0.13%  (lower = stronger attack)
  mean Linf perturbation: 0.2508
  kept (clean-ok & noise-ok & adv-fooled): 9105 / 10000
saved -> /kaggle/working/CIFAR10/BIM
[cifar10 / DeepFool]
  clean acc  : 93.67%
  noisy acc  : 92.26%
  adv   acc  : 0.32%  (lower = stronger attack)
  mean Linf perturbation: 0.3613
  kept (clean-ok & noise-ok & adv-fooled): 9092 / 10000
saved -> /kaggle/working/CIFAR10/DeepFool
[cifar10 / CWL2]
  clean acc  : 93.67%
  noisy acc  : 93.48%
  adv   acc  : 7.95%  (lower = stronger attack)
  mean Linf perturbation: 0.0981
  kept (clean-ok & noise-ok & adv-fooled): 8514 / 10000
saved -> /kaggle/working/CIFAR10/CWL2
